# Day 8 — Fine-Tuning with Hugging Face
## 30 Days of AI: From NLP to LLMs

---

On Day 7 you learned to use pretrained models as-is: load, tokenize,
run inference. This works for general tasks. But what happens when
your problem is specific — medical notes, legal documents, customer
support tickets in a niche domain?

The answer is fine-tuning: take a pretrained model that already
understands language, then train it further on your specific task
and data. The pretrained weights are your starting point — you are
not learning language from scratch, you are redirecting existing
knowledge toward a new objective.

Today you fine-tune BERT for text classification using Hugging Face
Trainer — the standard training loop used across industry and research.
You also explore the datasets library, evaluation metrics, and the
modern PEFT/LoRA approach for fine-tuning large models efficiently.

---

### What You Will Learn Today

- Why fine-tuning works — transfer learning intuition
- The `datasets` library — loading and preprocessing standard benchmarks
- PEFT/LoRA — fine-tune a 7B model on a single GPU
- When to fine-tune vs when to use prompting

### Goal by End of Day

Fine-tune DistilBERT on the IMDb sentiment dataset from scratch.
Evaluate it with accuracy and F1. Understand what LoRA does and
why it has become the default approach for large model fine-tuning.

In [7]:
## Run once
#!pip install transformers datasets evaluate torch scikit-learn -q
import warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Device  :', DEVICE)
print('PyTorch :', torch.__version__)

import transformers, datasets
print('Transformers :', transformers.__version__)
print('Datasets     :', datasets.__version__)

Device  : cpu
PyTorch : 2.10.0+cpu
Transformers : 4.46.3
Datasets     : 4.6.1


---

## Part 1 — Why Fine-Tuning Works: Transfer Learning

BERT was pretrained on Wikipedia (2.5B words) and BookCorpus (800M words)
using two self-supervised objectives:

```
Masked Language Modeling (MLM)
  Input  : 'The [MASK] sat on the mat'
  Target : predict 'cat'
  Effect : model learns word meaning and context

Next Sentence Prediction (NSP)
  Input  : Sentence A + Sentence B
  Target : does B follow A? (binary)
  Effect : model learns cross-sentence relationships
```

After pretraining, BERT's 110M parameters encode rich knowledge about
English. Fine-tuning repurposes this knowledge.

### What Changes During Fine-Tuning

```
Full Fine-Tuning
  → Add a task head on top of the pretrained model
  → Update ALL parameters via gradient descent
  → Pros  : maximum flexibility, best performance on small datasets
  → Cons  : expensive (GPU RAM = full model), catastrophic forgetting risk

Feature Extraction
  → Freeze all pretrained weights (requires_grad=False)
  → Only train the task head
  → Pros  : fast, minimal GPU RAM, preserves pretrained representations
  → Cons  : representations may not be optimal for your task

PEFT / LoRA (Parameter-Efficient Fine-Tuning)
  → Freeze pretrained weights
  → Add small trainable adapter layers or low-rank matrices
  → Train only 0.1% to 1% of total parameters
  → Pros  : fine-tune 7B+ models on a single consumer GPU
  → Cons  : slight performance gap vs full fine-tuning on small models
```

### The Layer Intuition

```
BERT Layer 1-3   : syntax, grammar, token-level patterns
BERT Layer 4-8   : semantics, word sense, entity types
BERT Layer 9-12  : task-specific, high-level representations

For most fine-tuning tasks, early layers are already good.
Fine-tuning mostly adjusts the top layers toward your task.
```

---

## Part 2 — The datasets Library

The `datasets` library provides a uniform interface to thousands of
standard NLP benchmarks. Datasets are memory-mapped using Apache Arrow —
you can work with datasets larger than your RAM.

### Loading and Inspecting a Dataset

```
load_dataset(path, name=None, split=None)

path   →  dataset identifier on the Hub (e.g. 'imdb', 'glue', 'squad')
name   →  subset (e.g. name='sst2' for GLUE's SST-2 split)
split  →  'train', 'test', 'validation', or a slice like 'train[:1000]'

Returns DatasetDict with named splits, or Dataset for a single split.
```

### Key Dataset Operations

```
dataset.map(function)           →  apply a function to all examples
dataset.filter(function)        →  keep only examples where function is True
dataset.shuffle(seed=42)        →  randomize order reproducibly
dataset.select(range(1000))     →  take a slice by index
dataset.train_test_split(0.1)   →  create a validation set
dataset.set_format('torch')     →  return PyTorch tensors from __getitem__
```

In [8]:
from datasets import load_dataset

# ---------------------------------------------------------------
# Load IMDb — 25,000 train, 25,000 test movie reviews
# Label 0 = negative,  Label 1 = positive
# ---------------------------------------------------------------

raw = load_dataset('imdb')

print('Dataset Structure')
print('=' * 55)
print(raw)
print()

print('Column names :', raw['train'].column_names)
print('Features     :', raw['train'].features)
print()

# Each example is a dictionary
example = raw['train'][0]
print('Example review (first 200 chars):')
print(example['text'][:200])
print(f'\nLabel : {example["label"]}  (0=negative, 1=positive)')

Dataset Structure
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

Column names : ['text', 'label']
Features     : {'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}

Example review (first 200 chars):
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev

Label : 0  (0=negative, 1=positive)


In [9]:
from transformers import AutoTokenizer

# ---------------------------------------------------------------
# Tokenize the dataset using .map()
# batched=True processes multiple examples at once — much faster
# We use DistilBERT: a 40% smaller, 60% faster version of BERT
# DistilBERT retains 97% of BERT's performance via knowledge distillation
# ---------------------------------------------------------------

MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding    = True,
        truncation = True,
        max_length = 256,   # IMDb reviews can be very long — truncate
    )

# Use small subsets for fast experimentation
# Remove these .select() calls for full training
train_raw = raw['train'].shuffle(seed=42).select(range(2000))
test_raw  = raw['test'].shuffle(seed=42).select(range(500))

tokenized_train = train_raw.map(tokenize, batched=True)
tokenized_test  = test_raw.map(tokenize,  batched=True)

# Remove the raw text column — model only needs token IDs
tokenized_train = tokenized_train.remove_columns(['text'])
tokenized_test  = tokenized_test.remove_columns(['text'])

# Rename label → labels (Trainer expects this name)
tokenized_train = tokenized_train.rename_column('label', 'labels')
tokenized_test  = tokenized_test.rename_column('label', 'labels')

# Return PyTorch tensors
tokenized_train.set_format('torch')
tokenized_test.set_format('torch')

print('Tokenized Dataset')
print('=' * 55)
print('Train columns :', tokenized_train.column_names)
print('Train size    :', len(tokenized_train))
print('Test size     :', len(tokenized_test))
print()
print('First example keys and shapes:')
for k, v in tokenized_train[0].items():
    print(f'  {k:<20} : shape {v.shape}')

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenized Dataset
Train columns : ['labels', 'input_ids', 'attention_mask']
Train size    : 2000
Test size     : 500

First example keys and shapes:
  labels               : shape torch.Size([])
  input_ids            : shape torch.Size([256])
  attention_mask       : shape torch.Size([256])


---

## Part 3 — LoRA: Fine-Tuning Large Models Efficiently

Full fine-tuning BERT (110M params) fits on any modern GPU.
Full fine-tuning Llama-2-7B (7B params) requires 28GB+ of GPU RAM
for the weights alone — before gradients and optimizer states.

LoRA (Low-Rank Adaptation) solves this by observing that the weight
updates during fine-tuning have low intrinsic rank.

### The LoRA Idea

```
Original weight matrix  :  W  ∈ R^(d x d)       ← frozen, not updated
LoRA adds               :  ΔW = B × A

  A  ∈ R^(r x d)   where r << d   (e.g. r=8, d=4096)
  B  ∈ R^(d x r)

Forward pass becomes    :  h = Wx + BAx
                                 ↑     ↑
                              frozen  trainable

Parameters trained:
  Original  :  d × d       =  4096 × 4096  =  16,777,216
  LoRA      :  2 × r × d   =  2 × 8 × 4096 =       65,536
  Reduction :  99.6% fewer parameters to train
```

After training, A and B are merged back into W:
`W_new = W + BA` — zero inference overhead.

### LoRA Hyperparameters

```
r       →  rank of the update matrices (4, 8, 16, 32, 64)
           higher r = more capacity but more parameters

alpha   →  scaling factor, usually set to r or 2r
           actual update = (alpha / r) × BA

target_modules  →  which weight matrices to adapt
               →  typically query and value projection matrices
               →  ['q_proj', 'v_proj'] or all attention projections

dropout →  regularization on the LoRA layers (0.05 to 0.1)
```

In [12]:
## !pip install peft -q

# ---------------------------------------------------------------
# Demonstrate LoRA concept manually on a small matrix
# This shows exactly what LoRA does mathematically
# ---------------------------------------------------------------

torch.manual_seed(42)

d = 64   # original weight dimension
r = 4    # LoRA rank — much smaller than d

W = torch.randn(d, d)          # pretrained weight matrix (frozen)
A = torch.randn(r, d) * 0.01  # LoRA A — initialized with small noise
B = torch.zeros(d, r)          # LoRA B — initialized to ZERO
                                # Zero B means ΔW=BA=0 at init
                                # → model behavior unchanged at start

x = torch.randn(d)             # input vector

# Forward pass
out_original = W @ x            # original forward pass
out_lora     = W @ x + B @ A @ x  # LoRA forward pass

print('LoRA Concept Demonstration')
print('=' * 55)
print(f'Original weight W  :  {d} x {d}  =  {d*d:,} parameters')
print(f'LoRA matrices A+B  :  {r}x{d} + {d}x{r}  =  {2*r*d:,} parameters')
print(f'Reduction          :  {(1 - 2*r*d/(d*d))*100:.1f}% fewer params')
print()
print(f'At initialization (B=0):')
print(f'  original output norm  : {out_original.norm().item():.4f}')
print(f'  lora output norm      : {out_lora.norm().item():.4f}')
print(f'  difference            : {(out_lora - out_original).norm().item():.6f}')
print()
print('Difference is zero at init — LoRA starts identical to original.')
print('As training progresses, B learns and ΔW = BA grows.')

LoRA Concept Demonstration
Original weight W  :  64 x 64  =  4,096 parameters
LoRA matrices A+B  :  4x64 + 64x4  =  512 parameters
Reduction          :  87.5% fewer params

At initialization (B=0):
  original output norm  : 53.3803
  lora output norm      : 53.3803
  difference            : 0.000000

Difference is zero at init — LoRA starts identical to original.
As training progresses, B learns and ΔW = BA grows.


In [13]:
# ---------------------------------------------------------------
# Apply PEFT LoRA to DistilBERT and count trainable parameters
# In production you would then train with Trainer just like before
# ---------------------------------------------------------------

try:
    from peft import LoraConfig, get_peft_model, TaskType

    # Fresh model — not the fine-tuned one
    base_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    )

    lora_config = LoraConfig(
        r              = 8,
        lora_alpha     = 16,
        target_modules = ['q_lin', 'v_lin'],  # DistilBERT's attention projections
        lora_dropout   = 0.1,
        bias           = 'none',
        task_type      = TaskType.SEQ_CLS,
    )

    lora_model = get_peft_model(base_model, lora_config)

    total_params    = sum(p.numel() for p in lora_model.parameters())
    trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)

    print('LoRA Applied to DistilBERT')
    print('=' * 55)
    print(f'Total parameters      : {total_params:,}')
    print(f'Trainable parameters  : {trainable_params:,}')
    print(f'Trainable percentage  : {100 * trainable_params / total_params:.2f}%')
    print()
    print('Only the LoRA A and B matrices are trained.')
    print('All original DistilBERT weights are frozen.')
    print()
    print('For a 7B model this makes the difference between:')
    print('  Full fine-tuning  →  80GB+ GPU (A100 80GB, ~$3/hr)')
    print('  LoRA fine-tuning  →  16GB GPU  (RTX 4090, consumer card)')

except ImportError:
    print('peft not installed. Run: pip install peft')
    print()
    print('LoRA concept: freeze original weights W, train only B and A')
    print('  W stays frozen  :  ~66M params NOT trained')
    print('  LoRA A + B      :  ~200K params trained')
    print('  Trainable %     :  ~0.3%')

peft not installed. Run: pip install peft

LoRA concept: freeze original weights W, train only B and A
  W stays frozen  :  ~66M params NOT trained
  LoRA A + B      :  ~200K params trained
  Trainable %     :  ~0.3%


---

## Day 8 Summary

```
What you built today:

1.  Datasets            →  load_dataset, .map(), padding, formatting
2.  LoRA                →  math + PEFT implementation

The two-day Hugging Face arc:

  Day 7  :  Use models as-is — pipelines, tokenizers, AutoModel
  Day 8  :  Adapt models — fine-tuning, evaluation, LoRA, deployment

  Together these cover ~90% of what practitioners do with Hugging Face
  in real projects.

```
### Self-Check Questions

Answer these before Day 9:

1. What is the difference between full fine-tuning and feature extraction?
2. Why does LoRA initialize matrix B to zeros?
3. What does `warmup_ratio=0.1` do to the learning rate schedule?
4. When would you choose prompting over fine-tuning?
5. If DistilBERT has 66M parameters and you apply LoRA with r=8 to
   the query and value projections of all 6 layers, roughly how many
   parameters are trainable?

### 30-Day AI Journey — Where You Are